# Project 2 - Deep Learning Based Arabic Audio Understanding and Retrieval System

## Student Credentials:
- Name: Yousef Ibrahim Gomaa Mahmoud
- ID: 320210207

In [1]:
!wget https://en.arabicspeechcorpus.com/arabic-speech-corpus.zip

!unzip -q arabic-speech-corpus.zip -d ./arabic_corpus

--2026-05-09 21:05:12--  https://en.arabicspeechcorpus.com/arabic-speech-corpus.zip
Resolving en.arabicspeechcorpus.com (en.arabicspeechcorpus.com)... 162.55.179.156
Connecting to en.arabicspeechcorpus.com (en.arabicspeechcorpus.com)|162.55.179.156|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1192302846 (1.1G) [application/zip]
Saving to: ‘arabic-speech-corpus.zip’

arabic-speech-corpu 100%[===================>]   1.11G  21.4MB/s    in 55s     

2026-05-09 21:06:07 (20.8 MB/s) - ‘arabic-speech-corpus.zip’ saved [1192302846/1192302846]



In [2]:
!pip -q install -U transformers accelerate librosa soundfile pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [39]:
import torch
import librosa
from transformers import ClapModel, ClapProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ClapModel.from_pretrained("laion/larger_clap_general").to(device)
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

def get_audio_embedding(audio_path):
    audio_sample, _ = librosa.load(audio_path, sr=48000)

    inputs = processor(audio=audio_sample, return_tensors="pt", sampling_rate=48000).to(device)

    with torch.no_grad():
        outputs = model.get_audio_features(**inputs)

        if hasattr(outputs, 'audio_embeds') and outputs.audio_embeds is not None:
            audio_embeds = outputs.audio_embeds
        elif hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            audio_embeds = outputs.pooler_output
        elif isinstance(outputs, tuple):
            audio_embeds = outputs[0]
        else:
            audio_embeds = outputs

    return audio_embeds / audio_embeds.norm(p=2, dim=-1, keepdim=True)

def get_text_embedding(text_query):
    inputs = processor(text=text_query, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)

        if hasattr(outputs, 'text_embeds') and outputs.text_embeds is not None:
            text_embeds = outputs.text_embeds
        elif hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            text_embeds = outputs.pooler_output
        elif isinstance(outputs, tuple):
            text_embeds = outputs[0]
        else:
            text_embeds = outputs

    return text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

In [49]:
from torch.nn.functional import cosine_similarity

dataset_audio_file = "/content/arabic_corpus/arabic-speech-corpus/test set/wav/ARA NORM  0001.wav"

dataset_emb = get_audio_embedding(dataset_audio_file)

search_query = "أتاحت للبائع المتجول أن يكون جاذبا للمواطن الأقل دخلا"
query_emb = get_text_embedding(search_query)

score = cosine_similarity(query_emb, dataset_emb)
print(f"Similarity Score: {score.item():.4f}")

Similarity Score: 0.3878


In [50]:
import os
import pandas as pd
from tqdm import tqdm

wav_directory = "/content/arabic_corpus/arabic-speech-corpus/wav"

data = []
all_files = [f for f in os.listdir(wav_directory) if f.endswith('.wav')]

print(f"Found {len(all_files)} .wav files. Generating embeddings...")

for file_name in tqdm(all_files):
    file_path = os.path.join(wav_directory, file_name)
    embedding = get_audio_embedding(file_path)

    if embedding is not None:
        data.append({
            "file_name": file_name,
            "embedding": embedding
        })

df = pd.DataFrame(data)
output_csv_path = "/content/arabic_audio_embeddings.csv"
df.to_csv(output_csv_path, index=False)

print(f"\nSuccess! Saved embeddings for {len(df)} files to {output_csv_path}")

Found 1813 .wav files. Generating embeddings...


100%|██████████| 1813/1813 [02:44<00:00, 11.03it/s]



Success! Saved embeddings for 1813 files to /content/arabic_audio_embeddings.csv
